<a href="https://colab.research.google.com/github/Harish848926/Puzzle_Game/blob/main/puzzle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""Text Puzzle Game Development using LLMs.ipynb

Automatically generated by Colab.

Original file is located at:
    https://colab.research.google.com/drive/1your_unique_drive_id

# Text Puzzle Game Development using LLMs

## Project Overview
This project explores the use of large language models (LLMs) to create engaging, dynamic text-based puzzle games. We'll implement a game with multiple puzzle types, using open-source LLMs for content generation.

## Table of Contents
1. Environment Setup and Dependencies
2. Introduction to Open-Source LLMs
3. Game Design and Architecture
4. Puzzle Generation with LLMs
5. Game Implementation
6. Evaluation Methods
7. Conclusion and Future Work
"""

# 1. Environment Setup and Dependencies
!pip install transformers torch sentencepiece accelerate
!pip install sentence-transformers  # For semantic similarity checking

# Import required libraries
import random
import re
import json
import time
from typing import List, Dict, Any, Optional, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np

# Set device (use GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. Introduction to Open-Source LLMs
"""
We'll use two open-source models:
1. GPT-2: Good for general text generation
2. Llama-2 (if accessible) or Mistral: For more sophisticated reasoning

Note: For actual deployment, you might need to adjust based on available resources.
"""

# Initialize models with lazy loading
class ModelManager:
    def __init__(self):
        self.models = {}
        self.semantic_model = None

    def load_simple_model(self):
        """Load GPT-2 for simpler puzzles"""
        if "simple" not in self.models:
            print("Loading GPT-2 model...")
            gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
            gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
            gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
            self.models["simple"] = (gpt2_model, gpt2_tokenizer)
        return self.models["simple"]

    def load_complex_model(self):
        """Load a more capable model for complex puzzles"""
        if "complex" not in self.models:
            print("Loading complex model...")
            try:
                # Try to load a more capable model
                complex_model_name = "mistralai/Mistral-7B-v0.1"  # Replace with your preferred model
                complex_tokenizer = AutoTokenizer.from_pretrained(complex_model_name)
                complex_model = AutoModelForCausalLM.from_pretrained(
                    complex_model_name,
                    torch_dtype=torch.float16,
                    device_map="auto",
                    load_in_4bit=True  # For memory efficiency
                )
                self.models["complex"] = (complex_model, complex_tokenizer)
                print("Loaded complex model successfully")
            except Exception as e:
                print(f"Could not load complex model: {e}")
                # Fall back to GPT-2
                self.models["complex"] = self.load_simple_model()
        return self.models["complex"]

    def load_semantic_model(self):
        """Load model for semantic similarity checking"""
        if self.semantic_model is None:
            print("Loading semantic similarity model...")
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
        return self.semantic_model

model_manager = ModelManager()

# 3. Game Design and Architecture
"""
Our text puzzle game will feature multiple puzzle types:
1. Word Riddles
2. Logic Puzzles
3. Word Association Puzzles
4. Story-based Puzzles
5. Cryptic Clues

The game will have:
- A scoring system
- Hint mechanisms
- Progressive difficulty
- Session persistence
"""

class TextPuzzleGame:
    def __init__(self):
        self.score = 0
        self.level = 1
        self.puzzles_solved = 0
        self.hints_used = 0
        self.current_puzzle = None
        self.puzzle_types = [
            "word_riddle",
            "logic_puzzle",
            "word_association",
            "story_puzzle",
            "cryptic_clue"
        ]
        self.puzzle_history = []

    def generate_puzzle(self, puzzle_type: Optional[str] = None) -> Dict[str, Any]:
        """Generate a puzzle of the specified type with quality check"""
        max_attempts = 3
        for attempt in range(max_attempts):
            if not puzzle_type:
                puzzle_type = random.choice(self.puzzle_types)

            if puzzle_type == "word_riddle":
                puzzle = self.generate_word_riddle()
            elif puzzle_type == "logic_puzzle":
                puzzle = self.generate_logic_puzzle()
            elif puzzle_type == "word_association":
                puzzle = self.generate_word_association()
            elif puzzle_type == "story_puzzle":
                puzzle = self.generate_story_puzzle()
            elif puzzle_type == "cryptic_clue":
                puzzle = self.generate_cryptic_clue()
            else:
                puzzle = self.generate_word_riddle()  # Default fallback

            # Quality check - ensure the puzzle has valid content
            if self.is_valid_puzzle(puzzle):
                return puzzle

        # If all attempts failed, return a simple fallback puzzle
        return {
            "type": "word_riddle",
            "question": "I'm tall when I'm young and short when I'm old. What am I?",
            "answer": "candle",
            "hints": ["It produces light", "It has a wick"],
            "difficulty": 1
        }

    def is_valid_puzzle(self, puzzle: Dict[str, Any]) -> bool:
        """Check if a puzzle meets quality standards"""
        if not puzzle or "question" not in puzzle or "answer" not in puzzle:
            return False

        # Check for minimum length and content
        if len(puzzle["question"].strip()) < 10:
            return False

        if len(puzzle["answer"].strip()) < 1:
            return False

        # Check if answer is actually contained in the question (a common LLM failure)
        if puzzle["answer"].lower() in puzzle["question"].lower():
            return False

        return True

    def generate_word_riddle(self) -> Dict[str, Any]:
        """Generate a word riddle using LLM"""
        prompt = """Create a clever word riddle with a single-word answer.
        Format: [RIDDLE] | [ANSWER]
        Example: I speak without a mouth and hear without ears. I have no body, but I come alive with the wind. What am I? | echo

        New riddle:"""

        result = self.generate_with_model(prompt, "simple")

        # Parse the result
        if "|" in result:
            riddle, answer = result.split("|", 1)
            answer = answer.strip().lower()
            # Remove any punctuation from answer
            answer = re.sub(r'[^\w\s]', '', answer)
        else:
            # Fallback if parsing fails
            riddle = result
            answer = self.extract_answer_from_riddle(riddle)

        return {
            "type": "word_riddle",
            "question": riddle,
            "answer": answer,
            "hints": [f"Starts with '{answer[0]}'", f"Has {len(answer)} letters"],
            "difficulty": min(3, self.level // 3 + 1)
        }

    def generate_logic_puzzle(self) -> Dict[str, Any]:
        """Generate a logic puzzle using LLM"""
        prompt = """Create a short logic puzzle with a clear solution.
        Format: [PUZZLE] | [SOLUTION]
        Example: There are three boxes labeled Apples, Oranges, and Mixed. Each label is incorrect. You can only take one fruit from one box. How do you determine the correct labels? | Take a fruit from the Mixed box. If it's an apple, then that box is actually Apples, the box labeled Oranges must be Mixed, and the box labeled Apples must be Oranges.

        New puzzle:"""

        result = self.generate_with_model(prompt, "complex")

        if "|" in result:
            puzzle, solution = result.split("|", 1)
        else:
            puzzle = result
            solution = "The solution requires logical deduction based on the given information."

        return {
            "type": "logic_puzzle",
            "question": puzzle,
            "answer": solution.lower(),
            "hints": ["Think step by step", "Consider all possibilities"],
            "difficulty": min(5, self.level // 2 + 2)
        }

    def generate_word_association(self) -> Dict[str, Any]:
        """Generate a word association puzzle"""
        words = ["time", "space", "light", "sound", "mind", "matter", "energy", "life"]
        word1, word2 = random.sample(words, 2)

        prompt = f"""Find a word that connects these two words: {word1} and {word2}.
        Format: [WORD1] - [CONNECTION] - [WORD2] | [CONNECTING WORD]
        Example: time - space - travel | travel
        Example: light - year - time | year

        New association: {word1} - {word2} |"""

        result = self.generate_with_model(prompt, "simple")
        connecting_word = result.strip().lower()

        return {
            "type": "word_association",
            "question": f"Find a word that connects '{word1}' and '{word2}'",
            "answer": connecting_word,
            "hints": [f"The word relates to both '{word1}' and '{word2}'", f"Has {len(connecting_word)} letters"],
            "difficulty": min(3, self.level // 4 + 1)
        }

    def generate_story_puzzle(self) -> Dict[str, Any]:
        """Generate a story-based puzzle"""
        themes = ["mystery", "adventure", "science fiction", "fantasy", "historical"]
        theme = random.choice(themes)

        prompt = f"""Create a very short {theme} story puzzle where the reader must determine what happened next or solve a mystery.
        End with a question that needs to be answered.
        Format: [STORY] | [ANSWER]
        Example: Detective Reed entered the room. The window was open, a chair was overturned, and a note was on the desk. The note read 'Midnight at the docks'. What should Detective Reed do next? | Go to the docks at midnight

        New story puzzle:"""

        result = self.generate_with_model(prompt, "complex")

        if "|" in result:
            story, answer = result.split("|", 1)
        else:
            story = result
            answer = self.extract_answer_from_story(story)

        return {
            "type": "story_puzzle",
            "question": story,
            "answer": answer.lower().strip(),
            "hints": ["Consider all clues in the story", "Think about what would make sense in this context"],
            "difficulty": min(4, self.level // 3 + 2)
        }

    def generate_cryptic_clue(self) -> Dict[str, Any]:
        """Generate a cryptic crossword-style clue"""
        words = ["apple", "river", "light", "sound", "dream", "cloud", "space", "time"]
        target_word = random.choice(words)

        prompt = f"""Create a cryptic crossword clue for the word '{target_word}'.
        Format: [CLUE] | [ANSWER]
        Example: Fruit that keeps the doctor away? | apple
        Example: What flows but isn't wet? | river

        New clue:"""

        result = self.generate_with_model(prompt, "simple")

        if "|" in result:
            clue, answer = result.split("|", 1)
            answer = answer.strip().lower()
        else:
            clue = result
            answer = target_word

        return {
            "type": "cryptic_clue",
            "question": clue,
            "answer": answer,
            "hints": [f"The answer has {len(answer)} letters", f"Rhymes with {self.find_rhyme(answer)}"],
            "difficulty": min(4, self.level // 3 + 1)
        }

    def generate_with_model(self, prompt: str, model_type: str = "simple", max_length: int = 150) -> str:
        """Generate text using the specified model"""
        try:
            if model_type == "simple":
                model, tokenizer = model_manager.load_simple_model()
            else:
                model, tokenizer = model_manager.load_complex_model()

            # Encode the prompt
            inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)

            # Generate response
            with torch.no_grad():
                outputs = model.generate(
                    inputs,
                    max_length=len(inputs[0]) + max_length,
                    num_return_sequences=1,
                    temperature=0.8,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id
                )

            # Decode and return the generated text
            generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Remove the prompt from the result
            result = generated[len(prompt):].strip()

            # Clean up the result
            result = re.sub(r'\[.*?\]', '', result)  # Remove any tags
            result = re.sub(r'\n+', ' ', result)  # Replace newlines with spaces

            return result
        except Exception as e:
            print(f"Error in generation: {e}")
            return "An error occurred while generating content."

    def extract_answer_from_riddle(self, riddle: str) -> str:
        """Extract a potential answer from a riddle"""
        # Simple heuristic: take the last word as answer
        words = riddle.split()
        if words:
            return words[-1].strip('?.!').lower()
        return "answer"

    def extract_answer_from_story(self, story: str) -> str:
        """Extract a potential answer from a story"""
        # Look for question words and take the next few words as answer
        question_words = ["what", "who", "where", "when", "why", "how"]
        words = story.lower().split()

        for i, word in enumerate(words):
            if word in question_words and i < len(words) - 1:
                # Return the next 2-3 words as potential answer
                return " ".join(words[i+1:i+4])

        # Fallback: return the last few words
        return " ".join(words[-3:])

    def find_rhyme(self, word: str) -> str:
        """Find a word that rhymes with the given word"""
        rhymes = {
            "apple": "chapel",
            "river": "shiver",
            "light": "night",
            "sound": "ground",
            "dream": "stream",
            "cloud": "proud",
            "space": "race",
            "time": "rhyme"
        }
        return rhymes.get(word, "something that rhymes")

    def check_answer(self, user_answer: str, puzzle: Dict[str, Any]) -> bool:
        """Check if the user's answer is correct using semantic similarity"""
        correct_answer = puzzle["answer"].lower()
        user_answer = user_answer.lower().strip()

        # Simple exact match
        if user_answer == correct_answer:
            return True

        # For logic puzzles and story puzzles, use semantic similarity
        if puzzle["type"] in ["logic_puzzle", "story_puzzle"]:
            similarity = self.semantic_similarity(user_answer, correct_answer)
            if similarity > 0.7:  # Threshold for semantic similarity
                return True

            # Also check for key terms in longer answers
            key_terms = correct_answer.split()[:5]  # First few words often contain key terms
            if any(term in user_answer for term in key_terms if len(term) > 4):
                return True

        # Partial match for longer answers
        if len(correct_answer) > 5 and correct_answer in user_answer:
            return True

        # Check if user answer contains the correct answer
        if correct_answer in user_answer:
            return True

        return False

    def semantic_similarity(self, text1: str, text2: str) -> float:
        """Calculate semantic similarity between two texts"""
        try:
            model = model_manager.load_semantic_model()
            embeddings = model.encode([text1, text2], convert_to_tensor=True)
            similarity = util.pytorch_cos_sim(embeddings[0], embeddings[1])
            return similarity.item()
        except Exception as e:
            print(f"Error calculating semantic similarity: {e}")
            return 0

    def get_hint(self, puzzle: Dict[str, Any]) -> str:
        """Get a hint for the current puzzle"""
        self.hints_used += 1

        if puzzle["hints"]:
            # Return the first hint and remove it from available hints
            return puzzle["hints"].pop(0)

        # Generate a generic hint if no specific hints are available
        if puzzle["type"] == "word_riddle":
            return f"The answer has {len(puzzle['answer'])} letters."
        elif puzzle["type"] == "logic_puzzle":
            return "Think step by step and consider all possibilities."
        else:
            return "Try thinking about the problem from a different angle."

    def solve_puzzle(self, user_answer: str) -> bool:
        """Attempt to solve the current puzzle"""
        if not self.current_puzzle:
            return False

        is_correct = self.check_answer(user_answer, self.current_puzzle)

        if is_correct:
            self.score += 10 * self.current_puzzle["difficulty"]
            self.puzzles_solved += 1
            self.level = self.puzzles_solved // 5 + 1
            self.puzzle_history.append({
                "puzzle": self.current_puzzle,
                "solved": True,
                "hints_used": 3 - len(self.current_puzzle["hints"]) if "hints" in self.current_puzzle else 0
            })
            return True
        else:
            self.score = max(0, self.score - 2)  # Small penalty for wrong answers
            return False

    def start_new_puzzle(self, puzzle_type: Optional[str] = None):
        """Generate and set a new puzzle"""
        self.current_puzzle = self.generate_puzzle(puzzle_type)
        return self.current_puzzle

# 5. Game Implementation
def play_game():
    """Main game loop"""
    game = TextPuzzleGame()
    print("Welcome to the Text Puzzle Game!")
    print("You'll solve puzzles generated by AI. Type 'hint' for a hint, 'skip' to skip, or 'quit' to exit.")

    while True:
        print(f"\n--- Level {game.level} | Score: {game.score} ---")
        puzzle = game.start_new_puzzle()

        print(f"\nPuzzle type: {puzzle['type'].replace('_', ' ').title()}")
        print(f"\n{puzzle['question']}")

        while True:
            user_input = input("\nYour answer: ").strip()

            if user_input.lower() == 'quit':
                print(f"Thanks for playing! Final score: {game.score}")
                return
            elif user_input.lower() == 'hint':
                hint = game.get_hint(puzzle)
                print(f"Hint: {hint}")
            elif user_input.lower() == 'skip':
                print(f"Skipped! The answer was: {puzzle['answer']}")
                game.puzzle_history.append({
                    "puzzle": puzzle,
                    "solved": False,
                    "hints_used": 3 - len(puzzle["hints"]) if "hints" in puzzle else 0
                })
                break
            else:
                if game.solve_puzzle(user_input):
                    print("Correct! Well done!")
                    break
                else:
                    print("Incorrect. Try again or ask for a hint.")

# 6. Evaluation Methods
def evaluate_game_performance(game_history: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Evaluate the performance of the game based on player history"""
    if not game_history:
        return {"error": "No game history provided"}

    total_puzzles = len(game_history)
    solved_puzzles = sum(1 for entry in game_history if entry["solved"])
    success_rate = solved_puzzles / total_puzzles * 100 if total_puzzles > 0 else 0

    total_hints = sum(entry["hints_used"] for entry in game_history)
    avg_hints_per_puzzle = total_hints / total_puzzles if total_puzzles > 0 else 0

    # Calculate difficulty success rates
    difficulty_stats = {}
    for entry in game_history:
        difficulty = entry["puzzle"]["difficulty"]
        if difficulty not in difficulty_stats:
            difficulty_stats[difficulty] = {"total": 0, "solved": 0}

        difficulty_stats[difficulty]["total"] += 1
        if entry["solved"]:
            difficulty_stats[difficulty]["solved"] += 1

    # Calculate success rate by puzzle type
    type_stats = {}
    for entry in game_history:
        p_type = entry["puzzle"]["type"]
        if p_type not in type_stats:
            type_stats[p_type] = {"total": 0, "solved": 0}

        type_stats[p_type]["total"] += 1
        if entry["solved"]:
            type_stats[p_type]["solved"] += 1

    return {
        "total_puzzles": total_puzzles,
        "solved_puzzles": solved_puzzles,
        "success_rate": success_rate,
        "total_hints_used": total_hints,
        "avg_hints_per_puzzle": avg_hints_per_puzzle,
        "difficulty_stats": difficulty_stats,
        "type_stats": type_stats
    }

def run_evaluation(num_puzzles: int = 10):
    """Run an automated evaluation of the puzzle generation"""
    game = TextPuzzleGame()
    evaluation_history = []

    print(f"Running evaluation with {num_puzzles} puzzles...")

    for i in range(num_puzzles):
        puzzle = game.start_new_puzzle()

        # Simulate a player with 70% chance of solving
        solved = random.random() < 0.7
        hints_used = random.randint(0, 3) if solved else 3

        evaluation_history.append({
            "puzzle": puzzle,
            "solved": solved,
            "hints_used": hints_used
        })

        print(f"Generated puzzle {i+1}/{num_puzzles}: {puzzle['type']}")

    # Evaluate performance
    results = evaluate_game_performance(evaluation_history)

    print("\n=== EVALUATION RESULTS ===")
    print(f"Total puzzles: {results['total_puzzles']}")
    print(f"Solved puzzles: {results['solved_puzzles']}")
    print(f"Success rate: {results['success_rate']:.2f}%")
    print(f"Total hints used: {results['total_hints_used']}")
    print(f"Average hints per puzzle: {results['avg_hints_per_puzzle']:.2f}")

    print("\nBy difficulty level:")
    for diff, stats in results['difficulty_stats'].items():
        rate = stats['solved'] / stats['total'] * 100 if stats['total'] > 0 else 0
        print(f"  Difficulty {diff}: {stats['solved']}/{stats['total']} ({rate:.2f}%)")

    print("\nBy puzzle type:")
    for p_type, stats in results['type_stats'].items():
        rate = stats['solved'] / stats['total'] * 100 if stats['total'] > 0 else 0
        print(f"  {p_type}: {stats['solved']}/{stats['total']} ({rate:.2f}%)")

    return results

# 7. Conclusion and Future Work
"""
This project demonstrates how LLMs can be used to create engaging text-based puzzle games.
Future improvements could include:
1. More sophisticated puzzle generation techniques
2. Better answer validation using semantic similarity
3. Adaptive difficulty based on player performance
4. Multiplayer functionality
5. Integration with voice assistants
6. Visual elements to complement the text puzzles
"""

# Save game state to Colab storage
def save_game_state(game, filename="puzzle_game_state.json"):
    """Save the current game state to a file"""
    game_data = {
        "score": game.score,
        "level": game.level,
        "puzzles_solved": game.puzzles_solved,
        "hints_used": game.hints_used,
        "puzzle_history": game.puzzle_history
    }

    with open(filename, 'w') as f:
        json.dump(game_data, f)

    print(f"Game state saved to {filename}")

# Load game state from Colab storage
def load_game_state(filename="puzzle_game_state.json"):
    """Load a game state from a file"""
    try:
        with open(filename, 'r') as f:
            game_data = json.load(f)

        game = TextPuzzleGame()
        game.score = game_data["score"]
        game.level = game_data["level"]
        game.puzzles_solved = game_data["puzzles_solved"]
        game.hints_used = game_data["hints_used"]
        game.puzzle_history = game_data["puzzle_history"]

        print("Game state loaded successfully!")
        return game
    except FileNotFoundError:
        print("No saved game found. Starting a new game.")
        return TextPuzzleGame()
    except Exception as e:
        print(f"Error loading game state: {e}")
        return TextPuzzleGame()

# Uncomment to play the game
# play_game()

# Uncomment to run evaluation
# evaluation_results = run_evaluation(15)

# Example of generating a single puzzle
game = TextPuzzleGame()
sample_puzzle = game.generate_puzzle("word_riddle")
print("Sample Puzzle:")
print(f"Type: {sample_puzzle['type']}")
print(f"Question: {sample_puzzle['question']}")
print(f"Answer: {sample_puzzle['answer']}")
print(f"Difficulty: {sample_puzzle['difficulty']}")

Using device: cpu
Loading GPT-2 model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Sample Puzzle:
Type: word_riddle
Question: What are you doing at the office? 
Answer: echo             echo             echo               echo                                                                        riddle 1 what is the name of your new name  echo 
Difficulty: 1


In [ ]:
# Add this section after the game implementation but before the conclusion

# Enhanced Input Handling System
class GameInputHandler:
    def __init__(self, game):
        self.game = game
        self.commands = {
            'help': self.show_help,
            'hint': self.get_hint,
            'skip': self.skip_puzzle,
            'quit': self.quit_game,
            'score': self.show_score,
            'history': self.show_history,
            'save': self.save_game,
            'load': self.load_game,
            'type': self.change_puzzle_type
        }

    def show_help(self):
        """Display available commands"""
        help_text = """
Available commands:
- Type your answer to solve the puzzle
- 'hint' - Get a hint for the current puzzle
- 'skip' - Skip the current puzzle
- 'quit' - Exit the game
- 'score' - Show your current score
- 'history' - Show your puzzle history
- 'save' - Save your game progress
- 'load' - Load a saved game
- 'type [puzzle_type]' - Change to a specific puzzle type
  Available types: riddle, logic, association, story, cryptic
- 'help' - Show this help message
"""
        print(help_text)
        return False  # Continue game

    def get_hint(self):
        """Get a hint for the current puzzle"""
        if self.game.current_puzzle:
            hint = self.game.get_hint(self.game.current_puzzle)
            print(f"Hint: {hint}")
        else:
            print("No active puzzle. Start a new puzzle first.")
        return False

    def skip_puzzle(self):
        """Skip the current puzzle"""
        if self.game.current_puzzle:
            print(f"Skipped! The answer was: {self.game.current_puzzle['answer']}")
            self.game.puzzle_history.append({
                "puzzle": self.game.current_puzzle,
                "solved": False,
                "hints_used": 3 - len(self.game.current_puzzle["hints"]) if "hints" in self.game.current_puzzle else 0
            })
            return True  # Signal to break current puzzle loop
        else:
            print("No active puzzle to skip.")
            return False

    def quit_game(self):
        """Exit the game"""
        print(f"Thanks for playing! Final score: {self.game.score}")
        return "quit"  # Special signal to quit game

    def show_score(self):
        """Display current score"""
        print(f"Current score: {self.game.score} | Level: {self.game.level} | Puzzles solved: {self.game.puzzles_solved}")
        return False

    def show_history(self):
        """Display puzzle history"""
        if not self.game.puzzle_history:
            print("No puzzle history yet.")
            return False

        print("\n--- Your Puzzle History ---")
        for i, entry in enumerate(self.game.puzzle_history, 1):
            status = "✓" if entry["solved"] else "✗"
            hints = f", Hints: {entry['hints_used']}" if entry['hints_used'] > 0 else ""
            print(f"{i}. {status} {entry['puzzle']['type'].replace('_', ' ').title()}{hints}")
        return False

    def save_game(self):
        """Save game state"""
        save_game_state(self.game)
        return False

    def load_game(self):
        """Load game state"""
        loaded_game = load_game_state()
        if loaded_game:
            self.game = loaded_game
            print("Game loaded successfully!")
        return False

    def change_puzzle_type(self, args):
        """Change the puzzle type"""
        if not args:
            print("Please specify a puzzle type. Available types: riddle, logic, association, story, cryptic")
            return False

        type_map = {
            'riddle': 'word_riddle',
            'logic': 'logic_puzzle',
            'association': 'word_association',
            'story': 'story_puzzle',
            'cryptic': 'cryptic_clue'
        }

        requested_type = args[0].lower()
        if requested_type in type_map:
            puzzle_type = type_map[requested_type]
            print(f"Changing to {requested_type} puzzles...")
            return puzzle_type
        else:
            print(f"Unknown puzzle type: {requested_type}. Available types: riddle, logic, association, story, cryptic")
            return False

    def process_input(self, user_input):
        """Process user input and execute appropriate command"""
        user_input = user_input.strip().lower()

        # Check if it's a command
        if user_input.startswith('/'):
            command_parts = user_input[1:].split()  # Remove the leading slash
            if command_parts:
                command = command_parts[0]
                args = command_parts[1:] if len(command_parts) > 1 else []

                if command in self.commands:
                    if command == 'type':
                        return self.commands[command](args)
                    else:
                        return self.commands[command]()
                else:
                    print(f"Unknown command: {command}. Type '/help' for available commands.")
                    return False
            else:
                print("Please specify a command after the slash. Type '/help' for available commands.")
                return False

        # Check if it's a regular answer attempt
        elif user_input:
            if self.game.current_puzzle:
                if self.game.solve_puzzle(user_input):
                    print("Correct! Well done!")
                    return True  # Signal to break current puzzle loop
                else:
                    print("Incorrect. Try again or ask for a hint with '/hint'")
                    return False
            else:
                print("No active puzzle. Start a new puzzle first.")
                return False

        # Empty input
        else:
            print("Please enter an answer or command. Type '/help' for available commands.")
            return False

# Enhanced Game Interface with Improved Input Handling
def play_game_enhanced():
    """Main game loop with enhanced input handling"""
    game = TextPuzzleGame()
    input_handler = GameInputHandler(game)

    print("=" * 50)
    print("        TEXT PUZZLE GAME USING LLMs")
    print("=" * 50)
    print("You'll solve AI-generated puzzles. Type '/help' for commands.")
    print("=" * 50)

    while True:
        print(f"\n--- Level {game.level} | Score: {game.score} | Solved: {game.puzzles_solved} ---")
        puzzle = game.start_new_puzzle()

        print(f"\nPuzzle type: {puzzle['type'].replace('_', ' ').title()} (Difficulty: {puzzle['difficulty']}/5)")
        print(f"\n{puzzle['question']}")

        # Puzzle solving loop
        while True:
            try:
                user_input = input("\nYour answer (or /command): ").strip()

                # Process input
                result = input_handler.process_input(user_input)

                if result == "quit":
                    return  # End game
                elif result is True:
                    break  # Correct answer or skip, move to next puzzle
                elif isinstance(result, str) and result.startswith("word_"):
                    # Changed puzzle type
                    puzzle = game.start_new_puzzle(result)
                    print(f"\nPuzzle type: {puzzle['type'].replace('_', ' ').title()} (Difficulty: {puzzle['difficulty']}/5)")
                    print(f"\n{puzzle['question']}")

            except KeyboardInterrupt:
                print("\n\nGame interrupted. Use '/quit' to exit properly.")
            except Exception as e:
                print(f"An error occurred: {e}")

# Additional utility functions for the input system
def get_validated_input(prompt, validation_func, error_message="Invalid input. Please try again."):
    """Get input with validation"""
    while True:
        try:
            user_input = input(prompt).strip()
            if validation_func(user_input):
                return user_input
            else:
                print(error_message)
        except KeyboardInterrupt:
            print("\nInput cancelled.")
            return None
        except Exception as e:
            print(f"Error: {e}")

# Add these functions to the existing save/load functions
def save_game_state(game, filename="puzzle_game_state.json"):
    """Save the current game state to a file"""
    game_data = {
        "score": game.score,
        "level": game.level,
        "puzzles_solved": game.puzzles_solved,
        "hints_used": game.hints_used,
        "puzzle_history": game.puzzle_history
    }

    with open(filename, 'w') as f:
        json.dump(game_data, f)

    print(f"Game state saved to {filename}")

def load_game_state(filename="puzzle_game_state.json"):
    """Load a game state from a file"""
    try:
        with open(filename, 'r') as f:
            game_data = json.load(f)

        game = TextPuzzleGame()
        game.score = game_data["score"]
        game.level = game_data["level"]
        game.puzzles_solved = game_data["puzzles_solved"]
        game.hints_used = game_data["hints_used"]
        game.puzzle_history = game_data["puzzle_history"]

        print("Game state loaded successfully!")
        return game
    except FileNotFoundError:
        print("No saved game found. Starting a new game.")
        return TextPuzzleGame()
    except Exception as e:
        print(f"Error loading game state: {e}")
        return TextPuzzleGame()

# Update the main execution section
if __name__ == "__main__":
    print("Select mode:")
    print("1. Play Game")
    print("2. Run Evaluation")
    print("3. Generate Sample Puzzle")

    choice = get_validated_input(
        "Enter your choice (1-3): ",
        lambda x: x in ['1', '2', '3'],
        "Please enter 1, 2, or 3."
    )

    if choice == '1':
        play_game_enhanced()
    elif choice == '2':
        num_puzzles = get_validated_input(
            "How many puzzles to evaluate? (default 10): ",
            lambda x: x.isdigit() or x == "",
            "Please enter a number."
        )
        count = int(num_puzzles) if num_puzzles.isdigit() else 10
        run_evaluation(count)
    elif choice == '3':
        game = TextPuzzleGame()
        sample_puzzle = game.generate_puzzle()
        print("\nSample Puzzle:")
        print(f"Type: {sample_puzzle['type'].replace('_', ' ').title()}")
        print(f"Question: {sample_puzzle['question']}")
        print(f"Answer: {sample_puzzle['answer']}")
        print(f"Difficulty: {sample_puzzle['difficulty']}/5")

Select mode:
1. Play Game
2. Run Evaluation
3. Generate Sample Puzzle
        TEXT PUZZLE GAME USING LLMs
You'll solve AI-generated puzzles. Type '/help' for commands.

--- Level 1 | Score: 0 | Solved: 0 ---
Loading complex model...
Could not load complex model: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/mistralai/Mistral-7B-v0.1.
401 Client Error. (Request ID: Root=1-68cc2ed0-3c7bc549108af85276f61a26;42c1d090-c7f2-4947-b916-18448a6efade)

Cannot access gated repo for url https://huggingface.co/mistralai/Mistral-7B-v0.1/resolve/main/config.json.
Access to model mistralai/Mistral-7B-v0.1 is restricted. You must have access to it and be authenticated to access it. Please log in.

Puzzle type: Story Puzzle (Difficulty: 2/5)

Detective Reed entered the room. The window was open, a chair was overturned, and a note was on the desk. The note read 'Midnight at the docks'. What should Detective Reed do next? 
Loading semantic similarity model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Incorrect. Try again or ask for a hint with '/hint'
Incorrect. Try again or ask for a hint with '/hint'
